In [8]:
from openai import OpenAI
from pinecone import Pinecone
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index = pc.Index("agent-rag")



In [9]:
def get_query_dense_embedding(query):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    return response.data[0].embedding
def get_query_sparse_embedding(query):

    response = pc.inference.embed(
        model="pinecone-sparse-english-v0",
        inputs=[query],
        parameters={
            "input_type": "query"
        }
    )

    sparse = response.data[0]

    return {
        "indices": sparse["sparse_indices"],
        "values": sparse["sparse_values"]
    }

In [10]:
def hybrid_search(query, top_k=5):

    # -------------------------
    # Dense Embedding
    # -------------------------

    dense_vector = get_query_dense_embedding(query)

    # -------------------------
    # Sparse Embedding
    # -------------------------

    # sparse_vector = get_query_sparse_embedding(query)

    # -------------------------
    # Hybrid Query
    # -------------------------

    results = index.query(

        vector=dense_vector,

        # sparse_vector=sparse_vector,

        top_k=top_k,

        include_metadata=True

    )

    return results

In [11]:
query = "what is llm"

results = hybrid_search(query)
print(results)

QueryResponse(matches=[ScoredVector(id='Understanding_Climate_Change.pdf_127', score=0.209262848, values=[], metadata={'captions': [], 'chunk_id': 127, 'doc_items': ['#/texts/295'], 'filename': 'Understanding_Climate_Change.pdf', 'headings': ['Land Rights and Protection'], 'origin_binary_hash': '8037439857228326505', 'origin_filename': 'Understanding_Climate_Change.pdf', 'origin_mimetype': 'application/pdf', 'page_numbers': ['20'], 'text': 'Land Rights and Protection\nSecuring land rights for indigenous and local communities is essential for climate justice. Recognizing and protecting these rights ensures that communities can manage their lands sustainably and resist exploitation. Legal frameworks and international agreements must uphold the rights of indigenous peoples.', 'title': 'Land Rights and Protection', 'token_count': 51}), ScoredVector(id='Understanding_Climate_Change.pdf_82', score=0.177578926, values=[], metadata={'captions': [], 'chunk_id': 82, 'doc_items': ['#/texts/183'],

In [12]:
results.matches

[ScoredVector(id='Understanding_Climate_Change.pdf_127', score=0.209262848, values=[], metadata={'captions': [], 'chunk_id': 127, 'doc_items': ['#/texts/295'], 'filename': 'Understanding_Climate_Change.pdf', 'headings': ['Land Rights and Protection'], 'origin_binary_hash': '8037439857228326505', 'origin_filename': 'Understanding_Climate_Change.pdf', 'origin_mimetype': 'application/pdf', 'page_numbers': ['20'], 'text': 'Land Rights and Protection\nSecuring land rights for indigenous and local communities is essential for climate justice. Recognizing and protecting these rights ensures that communities can manage their lands sustainably and resist exploitation. Legal frameworks and international agreements must uphold the rights of indigenous peoples.', 'title': 'Land Rights and Protection', 'token_count': 51}),
 ScoredVector(id='Understanding_Climate_Change.pdf_82', score=0.177578926, values=[], metadata={'captions': [], 'chunk_id': 82, 'doc_items': ['#/texts/183'], 'filename': 'Underst

## rank-bm25

In [13]:
pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import json

chunk_file = r"data/chunks/all_chunks.json"

with open(chunk_file, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Total chunks:", len(chunks))

Total chunks: 183


In [15]:
from rank_bm25 import BM25Okapi

documents = []
metadatas = []

for chunk in chunks:

    text = chunk["text"]

    documents.append(text)

    metadatas.append(chunk["metadata"])

# Tokenize
tokenized_docs = [
    doc.lower().split()
    for doc in documents
]

# Build BM25 index
bm25 = BM25Okapi(tokenized_docs)

print("BM25 model ready")

BM25 model ready


In [23]:
bm25

In [17]:
def bm25_search(query, top_k=5):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    # Get top indices
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []

    for idx in top_indices:

        results.append({
            "score": scores[idx],
            "text": documents[idx],
            "metadata": metadatas[idx]
        })

    return results

In [18]:
query = "economic resilience climate adaptation"

results = bm25_search(query, top_k=5)

In [19]:
for result in results:
    print(result)
    print("\n====================")

    print("Score:", result["score"])

    print("Title:",
          result["metadata"].get("title"))

    print("Filename:",
          result["metadata"].get("filename"))

    print("\nText:\n")

    print(result["text"])

{'score': np.float64(11.863136967425719), 'text': 'Economic Resilience\nBuilding economic resilience involves creating diverse and sustainable economies that can withstand climate impacts. This includes supporting small businesses, fostering innovation, and promoting local economic development. Resilient economies are better equipped to adapt to changing conditions and recover from disruptions.', 'metadata': {'filename': 'Understanding_Climate_Change.pdf', 'chunk_id': 136, 'page_numbers': [21], 'title': 'Economic Resilience', 'headings': ['Economic Resilience'], 'captions': [], 'doc_items': ['#/texts/317'], 'token_count': 51, 'origin': {'mimetype': 'application/pdf', 'binary_hash': 8037439857228326505, 'filename': 'Understanding_Climate_Change.pdf', 'uri': None}}}

Score: 11.863136967425719
Title: Economic Resilience
Filename: Understanding_Climate_Change.pdf

Text:

Economic Resilience
Building economic resilience involves creating diverse and sustainable economies that can withstand 

## sparse + Dense

In [20]:
def dense_search(query, top_k=10):

    dense_vector = get_query_dense_embedding(query)

    results = index.query(
        vector=dense_vector,
        top_k=top_k,
        include_metadata=True
    )

    formatted = []

    for rank, match in enumerate(results["matches"]):

        formatted.append({
            "id": match["id"],
            "rank": rank + 1,
            "score": match["score"],
            "text": match["metadata"]["text"],
            "metadata": match["metadata"]
        })

    return formatted
def bm25_search(query, top_k=10):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []

    for rank, idx in enumerate(top_indices):

        metadata = metadatas[idx]

        unique_id = (
            f"{metadata['filename']}_"
            f"{metadata['chunk_id']}"
        )

        results.append({
            "id": unique_id,
            "rank": rank + 1,
            "score": scores[idx],
            "text": documents[idx],
            "metadata": metadata
        })

    return results

In [32]:
from collections import defaultdict

def reciprocal_rank_fusion(

    dense_results,

    bm25_results,

    k=60
):

    fused_scores = defaultdict(float)

    chunk_data = {}

    # Dense Results
    for result in dense_results:

        doc_id = result["id"]

        rank = result["rank"]

        fused_scores[doc_id] += 1 / (k + rank)

        chunk_data[doc_id] = result

    # BM25 Results
    for result in bm25_results:

        doc_id = result["id"]

        rank = result["rank"]

        fused_scores[doc_id] += 1 / (k + rank)

        chunk_data[doc_id] = result

    ranked = sorted(

        fused_scores.items(),

        key=lambda x: x[1],

        reverse=True
    )

    final_results = []

    for doc_id, score in ranked:

        item = chunk_data[doc_id]

        item["fused_score"] = score

        final_results.append(item)

    return final_results

In [22]:
query = "economic resilience climate adaptation"

dense_results = dense_search(query)

bm25_results = bm25_search(query)

final_results = reciprocal_rank_fusion(
    dense_results,
    bm25_results
)

In [23]:
for result in final_results[:5]:

    print("\n====================")

    print("Fused Score:",
          result["fused_score"])

    print("Title:",
          result["metadata"]["title"])

    print(result["text"])


Fused Score: 0.03278688524590164
Title: Economic Resilience
Economic Resilience
Building economic resilience involves creating diverse and sustainable economies that can withstand climate impacts. This includes supporting small businesses, fostering innovation, and promoting local economic development. Resilient economies are better equipped to adapt to changing conditions and recover from disruptions.

Fused Score: 0.030798389007344232
Title: Public Health Infrastructure
Public Health Infrastructure
Strengthening public health infrastructure is vital for adapting to climate change. This includes enhancing healthcare facilities, improving disease surveillance systems, and training healthcare professionals. Community health programs can increase resilience and preparedness for climate-related health risks.

Fused Score: 0.030798389007344232
Title: Economic Impacts of Climate Change
Economic Impacts of Climate Change
The economic costs of climate change include damage to infrastructure,

## metadata filter

In [24]:
available_documents = [
    "NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.pdf",
    "Understanding_Climate_Change.pdf",
    "paper1.pdf"
]

In [25]:
# =====================================================
# OPENAI QUERY EMBEDDING
# =====================================================

def get_query_dense_embedding(query):

    response = client.embeddings.create(

        model="text-embedding-3-small",

        input=query
    )

    return response.data[0].embedding

In [26]:
import re

# -------------------------------------------------
# STOPWORDS
# -------------------------------------------------

STOPWORDS = {
    "the", "a", "an", "paper",
    "document", "pdf", "file",
    "explain", "what", "is",
    "from", "about", "of"
}


# -------------------------------------------------
# NORMALIZATION
# -------------------------------------------------

def normalize_text(text):

    text = text.lower()

    text = text.replace(".pdf", "")

    text = text.replace("-", " ")

    text = text.replace("_", " ")

    text = re.sub(r"[^a-z0-9\s]", "", text)

    return text


# -------------------------------------------------
# TOKEN CLEANING
# -------------------------------------------------

def clean_tokens(text):

    tokens = normalize_text(text).split()

    tokens = [
        token
        for token in tokens
        if token not in STOPWORDS
    ]

    return set(tokens)


# -------------------------------------------------
# DOCUMENT FILTER EXTRACTION
# -------------------------------------------------

def extract_metadata_filters(
    query,
    available_documents,
    overlap_threshold=2
):

    query_tokens = clean_tokens(query)

    matched_documents = []

    for doc in available_documents:

        doc_tokens = clean_tokens(doc)

        overlap = query_tokens.intersection(doc_tokens)

        print(f"\nDocument: {doc}")
        print("Overlap:", overlap)

        # -----------------------------------------
        # Threshold-based matching
        # -----------------------------------------

        if len(overlap) >= overlap_threshold:

            matched_documents.append(doc)

    # ---------------------------------------------
    # BUILD FILTER
    # ---------------------------------------------

    if matched_documents:

        return {
            "filename": {
                "$in": matched_documents
            }
        }

    return None

In [27]:
def dense_search(query, top_k=10):

    # -----------------------------------------
    # Dense Embedding
    # -----------------------------------------

    dense_vector = get_query_dense_embedding(query)

    # -----------------------------------------
    # Metadata Filter Extraction
    # -----------------------------------------

    metadata_filter = extract_metadata_filters(
        query=query,
        available_documents=available_documents
    )

    print("\nMetadata Filter:")
    print(metadata_filter)

    # -----------------------------------------
    # Pinecone Query
    # -----------------------------------------

    results = index.query(

        vector=dense_vector,

        top_k=top_k,

        include_metadata=True,

        filter=metadata_filter
    )

    formatted_results = []

    for rank, match in enumerate(results["matches"]):

        formatted_results.append({

            "id": match["id"],

            "rank": rank + 1,

            "retriever": "dense",

            "score": match["score"],

            "text": match["metadata"]["text"],

            "metadata": match["metadata"]
        })

    return formatted_results

In [29]:
query = """
Explain the AlexNet architecture
from the climage change paper
"""

results = dense_search(query)


Document: NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.pdf
Overlap: set()

Document: Understanding_Climate_Change.pdf
Overlap: {'change'}

Document: paper1.pdf
Overlap: set()

Metadata Filter:
None


In [35]:
query = "economic resilience climate adaptation"

dense_results = dense_search(query)

bm25_results = bm25_search(query)

final_results = reciprocal_rank_fusion(
    dense_results,
    bm25_results
)
print(final_results)


Document: NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.pdf
Overlap: set()

Document: Understanding_Climate_Change.pdf
Overlap: {'climate'}

Document: paper1.pdf
Overlap: set()

Metadata Filter:
None
[{'id': 'Understanding_Climate_Change.pdf_136', 'rank': 1, 'score': np.float64(11.863136967425719), 'text': 'Economic Resilience\nBuilding economic resilience involves creating diverse and sustainable economies that can withstand climate impacts. This includes supporting small businesses, fostering innovation, and promoting local economic development. Resilient economies are better equipped to adapt to changing conditions and recover from disruptions.', 'metadata': {'filename': 'Understanding_Climate_Change.pdf', 'chunk_id': 136, 'page_numbers': [21], 'title': 'Economic Resilience', 'headings': ['Economic Resilience'], 'captions': [], 'doc_items': ['#/texts/317'], 'token_count': 51, 'origin': {'mimetype': 'application/pdf', 'binary_hash': 8037439857228326

In [36]:
for a in final_results:
    print(a)

{'id': 'Understanding_Climate_Change.pdf_136', 'rank': 1, 'score': np.float64(11.863136967425719), 'text': 'Economic Resilience\nBuilding economic resilience involves creating diverse and sustainable economies that can withstand climate impacts. This includes supporting small businesses, fostering innovation, and promoting local economic development. Resilient economies are better equipped to adapt to changing conditions and recover from disruptions.', 'metadata': {'filename': 'Understanding_Climate_Change.pdf', 'chunk_id': 136, 'page_numbers': [21], 'title': 'Economic Resilience', 'headings': ['Economic Resilience'], 'captions': [], 'doc_items': ['#/texts/317'], 'token_count': 51, 'origin': {'mimetype': 'application/pdf', 'binary_hash': 8037439857228326505, 'filename': 'Understanding_Climate_Change.pdf', 'uri': None}}, 'fused_score': 0.03278688524590164}
{'id': 'Understanding_Climate_Change.pdf_99', 'rank': 7, 'score': np.float64(4.9843736748335), 'text': 'Public Health Infrastructure

## reranking with cohere

In [37]:
pip install cohere

   ---------------------------------------- 0.0/351.8 kB ? eta -:--:--
   ------------------- -------------------- 174.1/351.8 kB 3.5 MB/s eta 0:00:01
   ---------------------------------------  348.2/351.8 kB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 351.8/351.8 kB 3.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/441.1 kB ? eta -:--:--
   ------------------ --------------------- 204.8/441.1 kB 4.1 MB/s eta 0:00:01
   ---------------------------------------  440.3/441.1 kB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 441.1/441.1 kB 4.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cohere

co = cohere.Client("COHERE_API_KEY")

query = "What is attention mechanism?"

documents = [
    chunk["text"]
    for chunk in retrieved_chunks
]

response = co.rerank(
    model="rerank-english-v3.0",
    query=query,
    documents=documents,
    top_n=5
)

for result in response.results:
    print(result.index, result.relevance_score)